# Dialforge Cloud Benchmark v6

This launcher pins the source to one Git commit and forces the benchmark stack onto **managed Python 3.11**, regardless of the Python version Google Colab ships. It benchmarks Qwen 3 1.7B, 4B, 8B, faster-whisper small.en, Chatterbox Nano, and the full STT → LLM → TTS pipeline for all three Qwen tiers.

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all**.
3. Leave the tab open until the report appears.

Launcher: **v6-py311-commit-pinned**.

In [ ]:
# DIALFORGE BENCHMARK v6-py311-commit-pinned
import base64, json, os, pathlib, shutil, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

LAUNCHER_VERSION = 'v6-py311-commit-pinned'
OWNER = 'SumamaAhmed69'
REPO = 'Axemetric-Caller-Beta-Runtime'
BOOTSTRAP_PATH = 'benchmarks/dialforge_colab_bootstrap.py'
BOOTSTRAP = pathlib.Path('/content/dialforge_colab_bootstrap.py')
LOG = pathlib.Path('/content/dialforge-bootstrap.log')
REPORT = pathlib.Path('/content/dialforge-benchmark/dialforge-benchmark-report.html')

def api_json(url):
    req = urllib.request.Request(url, headers={
        'User-Agent': 'Dialforge-Colab-v6',
        'Accept': 'application/vnd.github+json',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    })
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.loads(response.read().decode('utf-8'))

print('Dialforge launcher:', LAUNCHER_VERSION, flush=True)
print('Resolving GitHub main to an immutable commit...', flush=True)
last_error = None
resolved_sha = None
payload = None
for attempt in range(1, 6):
    try:
        nonce = time.time_ns()
        commit = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/main?dialforge={nonce}')
        resolved_sha = commit['sha']
        print('Resolved main commit:', resolved_sha, flush=True)
        item = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{BOOTSTRAP_PATH}?ref={resolved_sha}&dialforge={nonce}')
        if item.get('encoding') != 'base64' or not item.get('content'):
            raise RuntimeError('GitHub Contents API did not return base64 bootstrap content')
        payload = base64.b64decode(item['content'])
        text = payload.decode('utf-8')
        required = ['bootstrap v5-py311', 'dialforge-benchmark-py311-v5', 'Provisioning managed CPython 3.11', 'uv', 'python_is_311']
        missing = [marker for marker in required if marker not in text]
        if missing:
            raise RuntimeError('Refusing incompatible bootstrap; missing markers: ' + ', '.join(missing))
        if 'dialforge-benchmark-venv-v3' in text:
            raise RuntimeError('Refusing stale bootstrap: obsolete venv-v3 marker found')
        compile(text, str(BOOTSTRAP), 'exec')
        BOOTSTRAP.write_bytes(payload)
        print(f'Bootstrap verified from commit {resolved_sha[:12]}: {len(payload)} bytes', flush=True)
        break
    except Exception as exc:
        last_error = exc
        print(f'GitHub API fetch attempt {attempt}/5 failed: {type(exc).__name__}: {exc}', flush=True)
        time.sleep(2 * attempt)
else:
    raise RuntimeError(f'Could not fetch a verified current Dialforge bootstrap: {last_error}')

for obsolete in [
    pathlib.Path('/content/dialforge-benchmark-venv-v3'),
    pathlib.Path('/content/dialforge-benchmark-venv-v4'),
]:
    if obsolete.exists():
        print('Removing obsolete environment:', obsolete, flush=True)
        shutil.rmtree(obsolete, ignore_errors=True)

print('\nStarting Python-3.11-pinned Dialforge bootstrap with LIVE output...', flush=True)
tail = []
env = os.environ.copy()
env['DIALFORGE_SOURCE_SHA'] = resolved_sha
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        [sys.executable, '-u', str(BOOTSTRAP)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
        tail.append(line)
        if len(tail) > 240:
            tail.pop(0)
    returncode = process.wait()

if returncode != 0:
    print('\n===== DIALFORGE FAILURE SUMMARY =====', flush=True)
    error_lines = [line for line in tail if ('ERROR' in line.upper() or 'RuntimeError' in line or 'Traceback' in line or 'failed' in line.lower())]
    if error_lines:
        print(''.join(error_lines[-60:]), flush=True)
    print('\n===== LAST BOOTSTRAP LINES =====', flush=True)
    print(''.join(tail[-120:]), flush=True)
    print('Resolved GitHub commit:', resolved_sha, flush=True)
    print('Full bootstrap log:', LOG, flush=True)
    raise RuntimeError(f'Dialforge bootstrap exited with code {returncode}. Exact failure is printed immediately above.')

if not REPORT.exists():
    raise RuntimeError(f'Bootstrap returned success but report is missing: {REPORT}')
print('\n=== DIALFORGE BENCHMARK COMPLETE ===', flush=True)
print('Source commit:', resolved_sha, flush=True)
display(HTML(REPORT.read_text(encoding='utf-8')))


### Output files
- `/content/dialforge-benchmark/dialforge-benchmark-report.html`
- `/content/dialforge-benchmark/dialforge-benchmark-report.json`
- `/content/dialforge-bootstrap.log`

The launcher refuses Python-3.13 benchmark environments and only continues once the managed environment is Python 3.11.